# 01 — Exploratory data analysis (AI4I / tractor sensors)

Point `DATA` at `../data/ai4i2020.csv` (Kaggle file or bootstrap output from `scripts/bootstrap_data.py`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path("../data/ai4i2020.csv")
df = pd.read_csv(DATA)
df.columns = [c.strip() for c in df.columns]
df.head()

In [ ]:
df.isna().sum()

In [ ]:
fail_cols = [c for c in ["TWF", "HDF", "PWF", "OSF", "RNF"] if c in df.columns]
df["failure_type"] = df.apply(
    lambda r: "HDF" if r.get("HDF", 0) == 1 else
              "PWF" if r.get("PWF", 0) == 1 else
              "OSF" if r.get("OSF", 0) == 1 else
              "TWF" if r.get("TWF", 0) == 1 else
              "RNF" if r.get("RNF", 0) == 1 else "Healthy",
    axis=1,
)
df["failure_type"].value_counts().plot(kind="bar", title="Counts by failure type")
plt.ylabel("count")
plt.tight_layout()

In [ ]:
df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power_kw"] = (df["Rotational speed [rpm]"] * df["Torque [Nm]"]) / 9549.0
num = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "temp_diff",
    "power_kw",
]
plt.figure(figsize=(10, 8))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f")
plt.title("Sensor correlation heatmap")
plt.tight_layout()

In [ ]:
sns.scatterplot(data=df.sample(min(2000, len(df))), x="Rotational speed [rpm]", y="Torque [Nm]", hue="failure_type", alpha=0.5)
plt.title("RPM vs torque (sample)")
plt.tight_layout()

In [ ]:
sns.histplot(df["temp_diff"], kde=True)
plt.title("Process − air temperature difference")
plt.tight_layout()